In [1]:
import GraphPCA as sg
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import squidpy as sq
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances as pair
from sklearn.metrics import adjusted_rand_score as ari_score

/home/vanhoan310/miniconda3/envs/biopy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/vanhoan310/miniconda3/envs/biopy311/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/home/vanhoan310/miniconda3/envs/biopy311/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [2]:
from sklearn.metrics.cluster import adjusted_rand_score
import numpy as np
import pandas as pd
import seaborn as sns
import os
import torch
import pandas as pd
from sklearn import metrics
import multiprocessing as mp
from sklearn.metrics.cluster import normalized_mutual_info_score

In [3]:
data_dir = '/data/hoan/spatial_transcriptomics/data/SDMBench/Data/378k/BrainAgingSpatialAtlas_MERFISH.h5ad'
adata=sc.read_h5ad(data_dir)

In [4]:
adata

AnnData object with n_obs × n_vars = 378918 × 374
    obs: 'fov', 'center_x', 'center_y', 'min_x', 'max_x', 'min_y', 'max_y', 'age', 'clust_annot', 'slice', 'organism_ontology_term_id', 'sex_ontology_term_id', 'suspension_type', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'donor_id', 'is_primary_data', 'cell_type_annot', 'tissue_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'batch_condition', 'citation', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_pca', 'X_spatial_coords', 'X_umap', 'spatial_coords'

In [5]:
adata.obs['slice_id'] = adata.obs['donor_id'].astype(str) +'Slice'+ adata.obs['slice'].astype(str)
adata.obsm['spatial'] = adata.obsm['spatial_coords']
adata.obs['batch'] = adata.obs['donor_id'].astype(str) +'Slice'+ adata.obs['slice'].astype(str)
adata.obs['Region'] = adata.obs['tissue']

In [6]:
output_folder = '/data/hoan/spatial_transcriptomics/GraphBG-analysis/output/graphpca/'

In [7]:
from GraphPCA import Run_Hierarchical_Multi_GPCA
# adata = sc.read_h5ad('/path/to/MERFISH_Aging.h5ad')
sections = [adata[adata.obs['slice_id'].eq(name)].copy() for name in adata.obs['slice_id'].unique()]
locations = [one.obsm['spatial'][:, :2] for one in sections]
Zs, W0, Ws, info = Run_Hierarchical_Multi_GPCA(
    sections, locations, n_components=30, lambdas=0.5, rhos=0.14872615,
    sample_weights='equal_slice', max_iter=50, random_seed=666, return_info=True, n_neighbors = 7)
print(info.converged, info.n_iter)

Reached maximum alternating iterations (50).


False 50


In [16]:
Z = np.vstack(Zs)

In [17]:
Z.shape

(378918, 30)

In [18]:
true_labels = adata.obs['Region'].values
estimator = KMeans(n_clusters=len(np.unique(true_labels)))
res = estimator.fit(Z)
lable_pred=res.labels_
print(normalized_mutual_info_score(true_labels, lable_pred))

0.5716711231180986


In [19]:
df_out = pd.DataFrame({'ground_truth': true_labels, 'GraphPCA': lable_pred})
df_out.to_csv('output/GraphPCA_BrainAgingSpatialAtlas.csv', index=False)